# CAASPP School-Level DFS — companion notebook

Calculates **Smarter Balanced-only DFS** (not official California Dashboard DFS). Upload districtwide **All Student Groups** files: 2021–23 caret-delimited, or 2024–26 caret/fixed-width TXT/ZIP files. Legacy 2022–23 fixed-width is also supported. All schools, including Mariposa, remain included. Older files may lack school names, so supply a newer file to backfill them automatically; otherwise codes are shown. 

Grade-level DFS = mean scale score minus grade/subject Level 3 cutoff; school DFS = student-with-scores weighted average. Check `Complete_3_5` before comparing three-grade school averages. Step 3 builds plots directly from the output School Code and School fields; it never guesses identities from historical chart colors.

References: [CAASPP 2026 format](https://caaspp-elpac.ets.org/caaspp/ResearchFileFormatSB?lstTestYear=2026), [CAASPP 2023 format](https://caaspp-elpac.ets.org/caaspp/ResearchFileFormatSB?lstTestYear=2023).


In [ ]:
#@title Step 1: Upload CAASPP TXT/CSV/ZIP files
from google.colab import files
uploaded=files.upload()
if not uploaded: raise ValueError('Please upload at least one research file')


In [ ]:
#@title Step 2: Calculate school-level DFS for legacy and newer files
import csv
from zipfile import ZipFile
from io import BytesIO, StringIO
import pandas as pd

CUTS = {'01': {'03':2432, '04':2473, '05':2502},
        '02': {'03':2436, '04':2485, '05':2528}}
SUBJECT = {'01':'ELA', '02':'Math'}
COUNTY_CODE = '19' #@param {type:'string'}
DISTRICT_CODE = '64683' #@param {type:'string'}

def f(line, a, b):
    return line[a-1:b].strip()

def inputs(uploaded):
    for name, blob in uploaded.items():
        if name.lower().endswith('.zip'):
            with ZipFile(BytesIO(blob)) as z:
                for member in z.namelist():
                    if member.lower().endswith(('.txt','.csv')) and not member.startswith('__MACOSX'):
                        yield member, z.read(member).decode('utf-8-sig', errors='replace')
        elif name.lower().endswith(('.txt','.csv')):
            yield name, blob.decode('utf-8-sig', errors='replace')
        else:
            raise ValueError(f'Unsupported file: {name}')

def get(row, *names):
    lower = {str(k).strip().lower().replace('_',' '): v for k,v in row.items()}
    for name in names:
        val = lower.get(name.lower())
        if val is not None:
            return str(val).strip()
    return ''

def parse_rows(source, content):
    lines = [s for s in content.splitlines() if s.strip()]
    if not lines:
        return
    first = lines[0]
    if '^' in first or (',' in first and not (len(first)>124 and f(first,121,124).isdigit())):
        sep = '^' if '^' in first else ','
        reader = csv.DictReader(StringIO(content), delimiter=sep)
        if not reader.fieldnames:
            raise ValueError(f'{source}: no header in delimited file')
        for row in reader:
            yield {
                'county':get(row,'County Code'),
                'district':get(row,'District Code'),
                'school_code':get(row,'School Code'),
                'school':get(row,'School Name'),
                'year':get(row,'Test Year'),
                'type':get(row,'Type ID'),
                'group':get(row,'Student Group ID','Subgroup ID'),
                'grade':get(row,'Grade'),
                'test':get(row,'Test ID'),
                'count':get(row,'Students with Scores','Total Students Tested with Scores',
                            'Total Tested with Scores'),
                'mean':get(row,'Mean Scale Score')
            }
    else:
        named = len(first)>=201 and f(first,121,124).isdigit()
        legacy = len(first)>=101 and f(first,19,22).isdigit()
        if not named and not legacy:
            raise ValueError(f'{source}: unknown fixed-width layout')
        for line in lines:
            if named:
                yield {
                    'county':f(line,1,2),'district':f(line,3,7),
                    'school_code':f(line,48,54),'school':f(line,55,114),
                    'type':f(line,115,116),'year':f(line,121,124),
                    'test':f(line,126,127),'group':f(line,128,130),
                    'grade':f(line,131,132),'count':f(line,151,159),
                    'mean':f(line,160,165)
                }
            else:
                yield {
                    'county':f(line,1,2),'district':f(line,3,7),
                    'school_code':f(line,8,14),'school':'',
                    'type':f(line,174,175) if len(line)>=175 else '',
                    'year':f(line,19,22),'group':f(line,23,25),
                    'grade':f(line,41,42),'test':f(line,43,44),
                    'count':f(line,95,101),'mean':f(line,59,64)
                }

rows = []
for source,content in inputs(uploaded):
    for r in parse_rows(source,content):
        if r['county'].zfill(2)!=COUNTY_CODE.zfill(2) or r['district'].zfill(5)!=DISTRICT_CODE.zfill(5):
            continue
        if r['group'].zfill(3)!='001':
            continue
        # Legacy files can omit Type ID; those rows are filtered by their school code.
        if r['type'] and r['type'].zfill(2) not in ('07','09','10'):
            continue
        if r['school_code'].zfill(7) in ('0000000', '1964683'):
            continue
        grade,test=r['grade'].zfill(2),r['test'].zfill(2)
        if grade not in ('03','04','05') or test not in CUTS:
            continue
        try:
            count=int(r['count'].replace(',',''))
            mean=float(r['mean'].replace(',',''))
            year=int(r['year'])
        except (ValueError,TypeError):
            continue  # suppressed/missing score records
        if count<=0 or not (2015<=year<=2030):
            continue
        rows.append({
            'Test Year':year,'School Code':r['school_code'].zfill(7),
            'School':r['school'],'Grade':int(grade),'Subject':SUBJECT[test],
            'Students With Scores':count,'Mean Scale Score':mean,
            'Level 3 Cutoff':CUTS[test][grade],
            'DFS':mean-CUTS[test][grade],'Source File':source
        })
detail=pd.DataFrame(rows)
if detail.empty:
    raise ValueError('No eligible school-level grade 3–5 records found. Check district codes and file type.')

# Older districtwide files may not embed school names. Fill from other years when possible.
name_lookup=(detail.loc[detail['School'].ne(''),['School Code','School']]
             .drop_duplicates('School Code').set_index('School Code')['School'].to_dict())
detail['School']=detail.apply(
    lambda r: r['School'] or name_lookup.get(r['School Code'], 'School '+r['School Code']),axis=1)

keys=['Test Year','School Code','Grade','Subject']
if detail.duplicated(keys).any():
    duplicates=detail.loc[detail.duplicated(keys,keep=False),keys+['Source File']]
    raise ValueError('Duplicate year/school/grade/subject rows:\n'+duplicates.to_string(index=False))

detail['Weighted DFS']=detail['DFS']*detail['Students With Scores']
school=(detail.groupby(['Test Year','School Code','School','Subject'],as_index=False)
        .agg(Students_With_Scores=('Students With Scores','sum'),
             Weighted_Total=('Weighted DFS','sum'),
             Grades_Included=('Grade','nunique')))
school['DFS']=(school['Weighted_Total']/school['Students_With_Scores']).round(1)
school['Complete_3_5']=school['Grades_Included'].eq(3)
school=school.drop(columns=['Weighted_Total']).sort_values(['Test Year','Subject','School'])
detail.drop(columns=['Weighted DFS']).to_csv('caaspp_dfs_grade_detail.csv',index=False)
school.to_csv('caaspp_dfs_school_summary.csv',index=False)
print('Calculated Smarter Balanced DFS only (not official Dashboard DFS). No school exclusions.')
print('Test years:',sorted(school['Test Year'].unique()))
display(school[['Test Year','School','Subject','Students_With_Scores','Grades_Included','Complete_3_5','DFS']])


### Validation using the 2022 LVUSD research file

Tested the caret-delimited file `sb_ca2022_all_19_64683_csv_v1.txt` (5,696 rows, 33 columns). All 10 school codes with grades 3–5 have usable ELA/Math mean scores, including `0140996` (which had suppressed scores in the supplied 2023 file). Calculated DFS values below are Smarter Balanced-only, not official Dashboard DFS.

| School code | 2022 ELA DFS | 2022 Math DFS |
|---|---:|---:|
| 0116129 | +16.4 | +6.2 |
| 0140996 | +17.5 | −17.1 |
| 6014799 | +58.6 | +34.8 |
| 6014807 | +47.2 | +35.9 |
| 6014815 | +52.7 | +38.4 |
| 6014823 | +44.3 | +26.7 |
| 6085658 | +33.0 | +15.4 |
| 6093025 | +45.2 | +7.5 |
| 6099030 | +57.6 | +40.5 |
| 6110563 | +25.6 | +9.4 |

2022 and 2023 records should be joined on school code, not the optional historical school name. Do not turn 2023 suppressed results into zeros.


### Validation using the 2023 LVUSD research file

Tested against `sb_ca2023_all_19_64683_csv_v1.txt` (a caret-delimited file despite its name). It has 33 columns and 6,004 rows. Group ID `1` is all students, and school type ID `7` is school-level. Nine school codes contain usable grade 3–5 mean scores for ELA/Math; one additional elementary school code (`0140996`) has scores suppressed (`*`) across these grades and is **not assigned a zero DFS**.

For regression checking, these reconstructed values rounded to one decimal place are expected:

| School code | 2023 ELA DFS | 2023 Math DFS |
|---|---:|---:|
| `0116129` | +8.1 | +4.0 |
| `6014799` | +53.9 | +21.7 |
| `6014807` | +54.3 | +38.0 |
| `6014815` | +60.5 | +55.2 |
| `6014823` | +31.8 | +22.2 |
| `6085658` | +35.4 | +28.0 |
| `6093025` | +45.0 | +11.9 |
| `6099030` | +52.4 | +33.1 |
| `6110563` | +18.9 | +5.2 |

These are calculated from rounded grade means, not verified official Dashboard figures. Missing/suppressed records must not be silently interpreted as zero, and a missing school is not an intentional exclusion.


In [ ]:
#@title Optional: Validate against the supplied 2022 LVUSD example
if set(school['Test Year']) == {2022} and COUNTY_CODE.zfill(2)=='19' and DISTRICT_CODE.zfill(5)=='64683':
    expected={('0116129','ELA'):16.4,('0116129','Math'):6.2,
              ('0140996','ELA'):17.5,('0140996','Math'):-17.1,
              ('6014815','ELA'):52.7,('6014815','Math'):38.4}
    observed=school.set_index(['School Code','Subject'])['DFS'].to_dict()
    for key,value in expected.items():
        assert observed.get(key)==value, f'2022 validation mismatch: {key}, expected {value}, got {observed.get(key)}'
    print('2022 sample checks passed; expected 10 school codes with usable scores.')
else:
    print('2022 sample-only check skipped when other years or districts are uploaded.')


In [ ]:
#@title Optional: Validate against the supplied 2023 LVUSD example
if set(school['Test Year']) == {2023} and COUNTY_CODE.zfill(2)=='19' and DISTRICT_CODE.zfill(5)=='64683':
    expected={('0116129','ELA'):8.1,('0116129','Math'):4.0,('6014815','ELA'):60.5,('6014815','Math'):55.2}
    observed=school.set_index(['School Code','Subject'])['DFS'].to_dict()
    for key,value in expected.items():
        assert observed.get(key)==value, f'2023 validation mismatch: {key}, expected {value}, got {observed.get(key)}'
    print('2023 sample checks passed; 9 school codes with usable ELA/Math data expected.')
else:
    print('Sample-only check skipped; the uploaded data include other years or districts.')


## School charts (no hard-coded school-code remapping)

Step 3 uses the school codes and names already extracted from CAASPP files. It raises an error for ambiguous school identities instead of silently assigning a chart to the wrong school. All schools with usable data—including Mariposa—are retained. The Excel workbook contains native charts that may be restyled when imported to Google Sheets, plus an audit table.


In [ ]:
#@title Step 3: Create correctly labeled ELA and Math charts + Google Sheets workbook
# Source of truth: the School Code and School columns produced by Step 2.
# No manually assigned school codes or color-based identity inference.
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

def normalize_school_name(value):
    s = re.sub(r"\\s+", " ", str(value)).strip()
    s = re.sub(r"\\s+(ELEMENTARY SCHOOL|ELEMENTARY|ES)$", "", s, flags=re.I)
    return s.title() if s else ""

chart_data = school.copy()
# Match schools across years by their stable school codes, *never* by colors or legend order.
recent_names = (chart_data.assign(has_name=~chart_data["School"].str.startswith("School "))
    .sort_values(["School Code", "has_name", "Test Year"])
    .groupby("School Code", as_index=False).tail(1))
name_lookup = dict(zip(recent_names["School Code"], recent_names["School"]))
chart_data["Display School"] = chart_data["School Code"].map(name_lookup).map(normalize_school_name)
chart_data["Display School"] = chart_data["Display School"].replace("", pd.NA).fillna(chart_data["School Code"])

# Refuse to silently merge two different school codes with the same display name.
collisions = chart_data.groupby("Display School")["School Code"].nunique()
if (collisions > 1).any():
    raise ValueError("Ambiguous school names map to multiple codes: " +
                     str(collisions[collisions > 1].to_dict()) +
                     ". Review the underlying school codes before plotting.")

# Refuse duplicate code / year / subject observations.
assert not chart_data.duplicated(["School Code", "Test Year", "Subject"]).any(), "Duplicate chart series records"
chart_data = chart_data.sort_values(["Subject", "Display School", "Test Year"])
chart_data.to_csv("caaspp_dfs_plot_audit.csv", index=False)

PALETTE = ["#2672C9", "#E3624A", "#D6A91C", "#279F75", "#8F5CC4",
           "#35A9B3", "#B55186", "#795548", "#7D9F30", "#9D6A37"]
for subject in ("ELA", "Math"):
    sub = chart_data[chart_data["Subject"] == subject]
    if sub.empty: continue
    series = sub.pivot(index="Test Year", columns="Display School", values="DFS").sort_index()
    fig, ax = plt.subplots(figsize=(13, 7.5), dpi=140)
    fig.patch.set_facecolor("white")
    vmin = min(-20, float(series.min().min()) - 8)
    vmax = max(75, float(series.max().max()) + 8)
    ax.axhspan(vmin, 0, facecolor="#FBEDB3", alpha=.65, zorder=0)
    ax.axhspan(0, 40, facecolor="#CAE2CB", alpha=.55, zorder=0)
    ax.axhspan(40, vmax, facecolor="#CFD3F9", alpha=.55, zorder=0)
    for i, name in enumerate(series.columns):
        y = series[name].dropna()
        if y.empty: continue
        ax.plot(y.index, y.values, marker="o", ms=5, lw=2.4,
                color=PALETTE[i % len(PALETTE)], label=name)
    ax.axhline(0, color="#41596B", lw=1.6)
    ax.set(title=f"{subject} • CAASPP Distance from Standard", ylabel="DFS (scale-score points)",
           xlabel="Test year", xlim=(series.index.min()-.15, series.index.max()+.15), ylim=(vmin,vmax))
    ax.set_xticks(series.index.tolist())
    ax.grid(axis="y", alpha=.2)
    ax.legend(loc="center left", bbox_to_anchor=(1.02,.5), frameon=False, fontsize=10)
    fig.subplots_adjust(right=.76)
    fig.text(.01,.015,"Reconstructed Smarter Balanced DFS (not official Dashboard DFS). Schools identified by CSV school code.",fontsize=9,color="#586475")
    fig.savefig(f"caaspp_dfs_{subject.lower()}_chart.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Native Excel line charts should import into Google Sheets; separate raw/audit sheets
# ensure the school identity can be checked regardless of chart import behavior.
output_path = "caaspp_dfs_charts_google_sheets.xlsx"
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    chart_data.to_excel(writer, sheet_name="Audit Data", index=False)
    for subject in ("ELA", "Math"):
        series = (chart_data[chart_data["Subject"] == subject]
                  .pivot(index="Test Year", columns="Display School", values="DFS")
                  .sort_index().reset_index())
        sheet_name = f"{subject} Wide"
        series.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1,1)
        ws.set_column(0,0,12)
        ws.set_column(1,len(series.columns)-1,21)
        chart = writer.book.add_chart({"type":"line"})
        for col in range(1,len(series.columns)):
            chart.add_series({
                "name":[sheet_name,0,col],
                "categories":[sheet_name,1,0,len(series),0],
                "values":[sheet_name,1,col,len(series),col],
                "marker":{"type":"circle","size":5},
                "line":{"color":PALETTE[(col-1)%len(PALETTE)],"width":2.25},
            })
        chart.set_title({"name":f"{subject} Distance From Standard"})
        chart.set_y_axis({"name":"DFS", "major_gridlines":{"visible":True}})
        chart.set_legend({"position":"right"})
        chart.set_size({"width":1080,"height":540})
        ws.insert_chart("L2",chart)
print("Created CSV audit, ELA/Math PNG charts, and Google Sheets-ready Excel chart workbook.")


In [ ]:
#@title Step 4: Download tables and charts
from google.colab import files
for path in ['caaspp_dfs_grade_detail.csv', 'caaspp_dfs_school_summary.csv',
             'caaspp_dfs_plot_audit.csv', 'caaspp_dfs_charts_google_sheets.xlsx',
             'caaspp_dfs_ela_chart.png', 'caaspp_dfs_math_chart.png']:
    if Path(path).exists(): files.download(path)


## Interpretation

Create trend charts from `caaspp_dfs_school_summary.csv` using Test Year on the x-axis and DFS on the y-axis. School names in old years come from any newer file uploaded in the same session. Missing grade coverage may distort comparisons. This notebook does not incorporate California Alternate Assessment results or Dashboard participation rules, and doesn't change the original percent-met notebook.
